<a href="https://colab.research.google.com/github/ys5591/Yunsu-Shin/blob/main/Copy_of_Working_Memory_Practice_Final_csv_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 0: Pre-processing code to perfectly parse and chunk the combined data (which has interspersed headers) by mouse/session into the list format required by the model

In [1]:
# 1. Cleanly uninstall existing conflicting packages (to prevent errors)
!pip uninstall -y ssm autograd scipy

# 2. Install autograd and the latest scipy compatible with the Python 3.12 environment
!pip install autograd scipy

# 3. [IMPORTANT] Install the author's custom ssm package directly from GitHub
!pip install git+https://github.com/Zeinab-Mohammadi/ssm.git

Found existing installation: autograd 1.8.0
Uninstalling autograd-1.8.0:
  Successfully uninstalled autograd-1.8.0
Found existing installation: scipy 1.17.1
Uninstalling scipy-1.17.1:
  Successfully uninstalled scipy-1.17.1
  Using cached autograd-1.8.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
Using cached autograd-1.8.0-py3-none-any.whl (51 kB)
Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.2 MB)
  Cloning https://github.com/Zeinab-Mohammadi/ssm.git to /tmp/pip-req-build-ymatpf5f
  Running command git clone --filter=blob:none --quiet https://github.com/Zeinab-Mohammadi/ssm.git /tmp/pip-req-build-ymatpf5f
  Resolved https://github.com/Zeinab-Mohammadi/ssm.git to commit 35377d405185b39be22ca459dbf5a66122e5d636
  Preparing metadata (setup.py) ... done
  Created wheel for ssm: filename=ssm-0.0.1-cp312-cp312-linux_x86_64.whl size=621441 sha256=16ab3b

In [2]:
# ============================================================
# Step 1. Load raw combined Excel and convert it to trial-level long format
# ============================================================

import os
import re
import sys
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import files


In [3]:
# ------------------------------------------------------------
# 1) Clone the author's GLM-HMM repository if it is not already in Colab
# ------------------------------------------------------------

REPO_DIR = Path("/content/glm-hmm_final")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/Zeinab-Mohammadi/glm-hmm_final.git", str(REPO_DIR)],
        check=True
    )

GLMHMM_CODE_DIR = REPO_DIR / "GLMHMM_and_GLM_frameworks"
sys.path.append(str(GLMHMM_CODE_DIR))

print("Repository ready:", REPO_DIR.exists())
print("GLM-HMM code path added:", GLMHMM_CODE_DIR)

Repository ready: True
GLM-HMM code path added: /content/glm-hmm_final/GLMHMM_and_GLM_frameworks


In [4]:
# ------------------------------------------------------------
# 2) Upload My raw combined Excel file
# ------------------------------------------------------------

print("Files currently in /content:")
print(os.listdir("/content"))

uploaded = files.upload()

xlsx_files = list(Path("/content").glob("*.xlsx"))

if len(xlsx_files) == 0:
    raise FileNotFoundError("No Excel file was found. Please upload your combined raw Excel file.")

raw_excel_path = xlsx_files[0]
print("Loaded raw Excel file:", raw_excel_path)

Files currently in /content:
['.config', 'glm-hmm_final', 'working_memory_long_format_step1.csv', 'Combined T maze recording.xlsx', 'wm_glmhmm_data', 'Combined T maze recording (3).xlsx', 'Combined T maze recording (4).xlsx', 'Combined T maze recording (2).xlsx', 'Combined T maze recording (1).xlsx', 'wm_glmhmm_results', 'sample_data']


Saving Combined T maze recording.xlsx to Combined T maze recording (5).xlsx
Loaded raw Excel file: /content/Combined T maze recording.xlsx


In [5]:
# ------------------------------------------------------------
# 3) Read the raw Excel file without assuming normal column headers
#    Your file has repeated blocks:
#    MouseID / Testing-Date row
#    Trials / Delay(s) / Order / Choice / Correct header row
#    trial rows
# ------------------------------------------------------------

raw = pd.read_excel(raw_excel_path, sheet_name=0, header=None)

print("Raw Excel shape:", raw.shape)
display(raw.head(15))

Raw Excel shape: (12960, 6)


,0,1,2,3,4,5
0,Sd43,Testing1-10,2025-08-18 00:00:00,NaN,NaN,NaN
1,Trials,Delay(s),Order,Choice,Correct,NaN
2,0,10,left,right,1,NaN
3,1,10,right,left,2,NaN
4,2,10,left,left,2,NaN
5,3,10,left,right,3,NaN
6,4,10,right,left,4,NaN
7,5,10,left,right,5,NaN
8,6,10,right,right,5,NaN
9,7,10,right,left,6,NaN


In [6]:
# ------------------------------------------------------------
# 4) Detect each block using the repeated 'Trials' header row
# ------------------------------------------------------------

header_rows = raw.index[
    raw[0].astype(str).str.strip().eq("Trials")
].tolist()

if len(header_rows) == 0:
    raise ValueError("No block headers were found. I expected rows where column 0 is 'Trials'.")

print("Number of detected session blocks:", len(header_rows))
print("First few header row indices:", header_rows[:5])

Number of detected session blocks: 405
First few header row indices: [1, 33, 65, 97, 129]


In [7]:
# ------------------------------------------------------------
# 5) Convert each block into a clean long-format dataframe
# ------------------------------------------------------------

all_blocks = []

for block_i, header_row in enumerate(header_rows):
    meta_row = header_row - 1

    mouse_id = str(raw.iloc[meta_row, 0]).strip()
    testing_label = str(raw.iloc[meta_row, 1]).strip()
    session_date = raw.iloc[meta_row, 2]

    # Extract testing number and delay from labels like Testing1-10
    match = re.search(r"Testing(\d+)-(\d+)", testing_label)

    if match is None:
        testing_num = np.nan
        delay_from_label = np.nan
    else:
        testing_num = int(match.group(1))
        delay_from_label = int(match.group(2))

    # Find where this block ends: right before the next header's metadata row
    if block_i < len(header_rows) - 1:
        next_meta_row = header_rows[block_i + 1] - 1
        block_end = next_meta_row
    else:
        block_end = len(raw)

    block = raw.iloc[header_row + 1:block_end, 0:5].copy()
    block.columns = ["trial_in_session", "delay_s", "order", "choice", "correct_cumulative"]

    # Keep only real trial rows
    block = block[pd.to_numeric(block["trial_in_session"], errors="coerce").notna()].copy()

    block["mouse_id"] = mouse_id
    block["testing_label"] = testing_label
    block["testing_num"] = testing_num
    block["session_date"] = session_date

    # Use delay from the actual column when possible, otherwise from Testing label
    block["delay_s"] = pd.to_numeric(block["delay_s"], errors="coerce")
    block["delay_s"] = block["delay_s"].fillna(delay_from_label).astype(int)

    block["trial_in_session"] = block["trial_in_session"].astype(int)

    block["order"] = block["order"].astype(str).str.strip().str.lower()
    block["choice"] = block["choice"].astype(str).str.strip().str.lower()

    # This is very important:
    # session_id keeps each mouse/session/delay as a separate behavioral sequence.
    # Do NOT delete mouse_id/session_id at this stage.
    block["session_id"] = (
        block["mouse_id"].astype(str)
        + "_"
        + block["testing_label"].astype(str)
        + "_"
        + pd.to_datetime(block["session_date"]).dt.strftime("%Y%m%d")
    )

    all_blocks.append(block)

df_long = pd.concat(all_blocks, ignore_index=True)

In [8]:
# ------------------------------------------------------------
# 6) Basic checks
# ------------------------------------------------------------

print("Clean long-format data shape:", df_long.shape)
print("Number of mice:", df_long["mouse_id"].nunique())
print("Number of sessions:", df_long["session_id"].nunique())
print("Delay values:", sorted(df_long["delay_s"].unique()))
print("Testing values:", sorted(df_long["testing_num"].dropna().unique()))

display(df_long.head(20))

# Save this intermediate clean file
clean_csv_path = Path("/content/working_memory_long_format_step1.csv")
df_long.to_csv(clean_csv_path, index=False)

print("Saved clean long-format CSV:", clean_csv_path)

Clean long-format data shape: (12150, 10)
Number of mice: 45
Number of sessions: 405
Delay values: [np.int64(10), np.int64(30), np.int64(60)]
Testing values: [np.int64(1), np.int64(2), np.int64(3)]


,trial_in_session,delay_s,order,choice,correct_cumulative,mouse_id,testing_label,testing_num,session_date,session_id
0,0,10,left,right,1,Sd43,Testing1-10,1,2025-08-18,Sd43_Testing1-10_20250818
1,1,10,right,left,2,Sd43,Testing1-10,1,2025-08-18,Sd43_Testing1-10_20250818
2,2,10,left,left,2,Sd43,Testing1-10,1,2025-08-18,Sd43_Testing1-10_20250818
3,3,10,left,right,3,Sd43,Testing1-10,1,2025-08-18,Sd43_Testing1-10_20250818
4,4,10,right,left,4,Sd43,Testing1-10,1,2025-08-18,Sd43_Testing1-10_20250818
5,5,10,left,right,5,Sd43,Testing1-10,1,2025-08-18,Sd43_Testing1-10_20250818
6,6,10,right,right,5,Sd43,Testing1-10,1,2025-08-18,Sd43_Testing1-10_20250818
7,7,10,right,left,6,Sd43,Testing1-10,1,2025-08-18,Sd43_Testing1-10_20250818
8,8,10,right,left,7,Sd43,Testing1-10,1,2025-08-18,Sd43_Testing1-10_20250818
9,9,10,left,right,8,Sd43,Testing1-10,1,2025-08-18,Sd43_Testing1-10_20250818


Saved clean long-format CSV: /content/working_memory_long_format_step1.csv


In [9]:
# ============================================================
# Step 2. Convert clean long-format data into GLM-HMM-ready arrays
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

In [10]:
# ------------------------------------------------------------
# 1) Load df_long if it is not already in memory
# ------------------------------------------------------------

if "df_long" not in globals():
    df_long = pd.read_csv("/content/working_memory_long_format_step1.csv")

df_model = df_long.copy()

# Sort data so trials are in the correct temporal order
df_model["session_date"] = pd.to_datetime(df_model["session_date"])

df_model = df_model.sort_values(
    ["mouse_id", "session_date", "delay_s", "testing_num", "trial_in_session"]
).reset_index(drop=True)

print("Data shape after sorting:", df_model.shape)
display(df_model.head())

Data shape after sorting: (12150, 10)


,trial_in_session,delay_s,order,choice,correct_cumulative,mouse_id,testing_label,testing_num,session_date,session_id
0,0,10,right,right,0,Sd101,Testing1-10,1,2025-10-11,Sd101_Testing1-10_20251011
1,1,10,right,left,1,Sd101,Testing1-10,1,2025-10-11,Sd101_Testing1-10_20251011
2,2,10,right,left,2,Sd101,Testing1-10,1,2025-10-11,Sd101_Testing1-10_20251011
3,3,10,left,right,3,Sd101,Testing1-10,1,2025-10-11,Sd101_Testing1-10_20251011
4,4,10,left,left,3,Sd101,Testing1-10,1,2025-10-11,Sd101_Testing1-10_20251011


In [11]:
# ------------------------------------------------------------
# 2) Clean and validate left/right variables
# ------------------------------------------------------------

df_model["order"] = df_model["order"].astype(str).str.lower().str.strip()
df_model["choice"] = df_model["choice"].astype(str).str.lower().str.strip()

valid_sides = {"left", "right"}

bad_order = df_model.loc[~df_model["order"].isin(valid_sides)]
bad_choice = df_model.loc[~df_model["choice"].isin(valid_sides)]

if len(bad_order) > 0:
    raise ValueError("Some order values are not left/right. Please check bad_order.")

if len(bad_choice) > 0:
    raise ValueError("Some choice values are not left/right. Please check bad_choice.")

print("Order and choice columns look valid.")


Order and choice columns look valid.


In [12]:
# ------------------------------------------------------------
# 3) Encode choice and task variables
# ------------------------------------------------------------

# output y:
# left choice  = 0
# right choice = 1
df_model["choice_right"] = (df_model["choice"] == "right").astype(int)

# order/sample side:
# left order  = 0
# right order = 1
df_model["order_right"] = (df_model["order"] == "right").astype(int)

# DNMS rule:
# correct choice is the opposite side of the sample/order side.
df_model["correct_side_right"] = 1 - df_model["order_right"]

# Trial-level correctness:
# Correct = 1 if choice is NOT equal to order.
# This is better than using your original Correct column directly because
# the original Correct column is cumulative.
df_model["trial_correct"] = (df_model["choice"] != df_model["order"]).astype(int)

# Signed coding:
# left  = -1
# right = +1
df_model["choice_signed"] = df_model["choice_right"] * 2 - 1
df_model["order_signed"] = df_model["order_right"] * 2 - 1
df_model["correct_side_signed"] = df_model["correct_side_right"] * 2 - 1

# Reward/correctness signed coding:
# incorrect = -1
# correct   = +1
df_model["reward_signed"] = df_model["trial_correct"] * 2 - 1

In [13]:
# ------------------------------------------------------------
# 4) Create previous-trial regressors within each session
# ------------------------------------------------------------

# For the first trial of each session, previous-trial variables are set to 0.
df_model["prev_choice_signed"] = (
    df_model.groupby("session_id")["choice_signed"]
    .shift(1)
    .fillna(0)
)

df_model["prev_correct_side_signed"] = (
    df_model.groupby("session_id")["correct_side_signed"]
    .shift(1)
    .fillna(0)
)

df_model["prev_reward_signed"] = (
    df_model.groupby("session_id")["reward_signed"]
    .shift(1)
    .fillna(0)
)


In [14]:
# ------------------------------------------------------------
# 5) Create delay and within-session time regressors
# ------------------------------------------------------------

# Standardized delay:
# This is important for your working-memory task because 10, 30, and 60 sec
# delays are different working-memory loads.
delay_mean = df_model["delay_s"].mean()
delay_std = df_model["delay_s"].std()

df_model["delay_z"] = (df_model["delay_s"] - delay_mean) / delay_std

# Trial progress within session:
# This is a simple version of the paper's warm-up / elapsed-time idea.
df_model["trial_progress"] = (
    df_model.groupby("session_id")["trial_in_session"]
    .transform(lambda x: x / x.max() if x.max() > 0 else 0)
)

trial_progress_mean = df_model["trial_progress"].mean()
trial_progress_std = df_model["trial_progress"].std()

df_model["trial_progress_z"] = (
    df_model["trial_progress"] - trial_progress_mean
) / trial_progress_std

In [15]:
# ------------------------------------------------------------
# 6) Build observation and transition design matrices
# ------------------------------------------------------------

# Observation GLM regressors:
# These regressors predict the animal's current left/right choice.
#
# Important:
# We include a final bias_placeholder column because the author's scripts
# expect to remove/replace the last column in some places.
obs_regressor_names = [
    "correct_side_signed",
    "prev_choice_signed",
    "prev_reward_signed",
    "delay_z",
    "bias_placeholder"
]

df_model["bias_placeholder"] = 1.0

obs_mat = df_model[obs_regressor_names].to_numpy(dtype=float)


# Transition GLM regressors:
# These regressors predict hidden-state transitions.
# They are allowed to differ from observation regressors.
trans_regressor_names = [
    "prev_choice_signed",
    "prev_correct_side_signed",
    "prev_reward_signed",
    "delay_z",
    "trial_progress_z",
    "transition_bias"
]

df_model["transition_bias"] = 1.0

trans_mat = df_model[trans_regressor_names].to_numpy(dtype=float)


# Output:
# The model predicts binary choice.
# Shape should be T x 1.
output = df_model["choice_right"].to_numpy(dtype=int).reshape(-1, 1)

# Session:
# This keeps each mouse/testing/delay block separate.
# This is essential. Do not delete this.
session = df_model["session_id"].astype(str).to_numpy()

In [16]:
# ------------------------------------------------------------
# 7) Basic shape and validity checks
# ------------------------------------------------------------

print("obs_mat shape:", obs_mat.shape)
print("trans_mat shape:", trans_mat.shape)
print("output shape:", output.shape)
print("session shape:", session.shape)

assert obs_mat.shape[0] == trans_mat.shape[0] == output.shape[0] == session.shape[0]
assert set(np.unique(output[:, 0])).issubset({0, 1})
assert np.isfinite(obs_mat).all()
assert np.isfinite(trans_mat).all()

print("All GLM-HMM arrays passed basic checks.")

print("\nOutput coding check:")
print(pd.Series(output[:, 0]).value_counts().sort_index())

print("\nFirst 5 rows of observation matrix:")
display(pd.DataFrame(obs_mat[:5], columns=obs_regressor_names))

print("\nFirst 5 rows of transition matrix:")
display(pd.DataFrame(trans_mat[:5], columns=trans_regressor_names))

obs_mat shape: (12150, 5)
trans_mat shape: (12150, 6)
output shape: (12150, 1)
session shape: (12150,)
All GLM-HMM arrays passed basic checks.

Output coding check:
0    6288
1    5862
Name: count, dtype: int64

First 5 rows of observation matrix:


,correct_side_signed,prev_choice_signed,prev_reward_signed,delay_z,bias_placeholder
0,-1.0,0.0,0.0,-1.134272,1.0
1,-1.0,1.0,-1.0,-1.134272,1.0
2,-1.0,-1.0,1.0,-1.134272,1.0
3,1.0,-1.0,1.0,-1.134272,1.0
4,1.0,1.0,1.0,-1.134272,1.0



First 5 rows of transition matrix:


,prev_choice_signed,prev_correct_side_signed,prev_reward_signed,delay_z,trial_progress_z,transition_bias
0,0.0,0.0,0.0,-1.134272,-1.675116,1.0
1,1.0,-1.0,-1.0,-1.134272,-1.559574,1.0
2,-1.0,-1.0,1.0,-1.134272,-1.444031,1.0
3,-1.0,-1.0,1.0,-1.134272,-1.328489,1.0
4,1.0,1.0,1.0,-1.134272,-1.212946,1.0


In [17]:
# ------------------------------------------------------------
# 8) Create session-level 5-fold cross-validation map
# ------------------------------------------------------------

# The paper uses 5-fold cross-validation at the session level.
# Here, folds are assigned within each mouse so each mouse contributes
# sessions across folds as evenly as possible.
rng = np.random.default_rng(0)

session_info = (
    df_model[["mouse_id", "session_id"]]
    .drop_duplicates()
    .sort_values(["mouse_id", "session_id"])
    .reset_index(drop=True)
)

fold_records = []

for mouse_id, sub in session_info.groupby("mouse_id"):
    mouse_sessions = sub["session_id"].to_numpy().copy()
    rng.shuffle(mouse_sessions)

    for i, sess in enumerate(mouse_sessions):
        fold_records.append([sess, i % 5])

fold_mapping_session = np.array(fold_records, dtype=object)

print("\nFold mapping shape:", fold_mapping_session.shape)
display(pd.DataFrame(fold_mapping_session, columns=["session_id", "fold"]).head(15))

print("\nFold counts:")
display(pd.DataFrame(fold_mapping_session, columns=["session_id", "fold"])["fold"].value_counts().sort_index())


Fold mapping shape: (405, 2)


,session_id,fold
0,Sd101_Testing2-30_20251015,0
1,Sd101_Testing2-60_20251018,1
2,Sd101_Testing1-60_20251017,2
3,Sd101_Testing3-10_20251013,3
4,Sd101_Testing2-10_20251012,4
5,Sd101_Testing3-60_20251019,0
6,Sd101_Testing3-30_20251016,1
7,Sd101_Testing1-10_20251011,2
8,Sd101_Testing1-30_20251014,3
9,Sd104_Testing1-60_20251017,0



Fold counts:


,count
fold,
0,90
1,90
2,90
3,90
4,45


In [18]:
# ------------------------------------------------------------
# 9) Save files in the structure expected by the GitHub code
# ------------------------------------------------------------

DATA_DIR = Path("/content/wm_glmhmm_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

combined_npz_path = DATA_DIR / "combined_all_mice.npz"
fold_npz_path = DATA_DIR / "fold_mapping_session_all_mice.npz"
metadata_csv_path = DATA_DIR / "working_memory_model_ready_metadata.csv"
regressor_names_path = DATA_DIR / "regressor_names.npz"

np.savez(
    combined_npz_path,
    obs_mat,
    trans_mat,
    output,
    session
)

np.savez(
    fold_npz_path,
    fold_mapping_session
)

np.savez(
    regressor_names_path,
    obs_regressor_names=np.array(obs_regressor_names),
    trans_regressor_names=np.array(trans_regressor_names)
)

df_model.to_csv(metadata_csv_path, index=False)

print("\nSaved model-ready data:")
print("combined npz:", combined_npz_path)
print("fold mapping npz:", fold_npz_path)
print("metadata csv:", metadata_csv_path)
print("regressor names:", regressor_names_path)


Saved model-ready data:
combined npz: /content/wm_glmhmm_data/combined_all_mice.npz
fold mapping npz: /content/wm_glmhmm_data/fold_mapping_session_all_mice.npz
metadata csv: /content/wm_glmhmm_data/working_memory_model_ready_metadata.csv
regressor names: /content/wm_glmhmm_data/regressor_names.npz


In [19]:
# ------------------------------------------------------------
# 4) Fit 1-state GLM separately for each held-out fold
# ------------------------------------------------------------

N_FOLDS = 5
prior_sigma = 100.0

glm_summary_rows = []

for fold in range(N_FOLDS):

    print("\n================================================")
    print(f"Fitting 1-state GLM pre-training model: fold {fold}")
    print("================================================")

    # Sessions NOT in this fold are training sessions
    train_sessions = fold_mapping_session[
        fold_mapping_session[:, 1].astype(int) != fold,
        0
    ].astype(str)

    test_sessions = fold_mapping_session[
        fold_mapping_session[:, 1].astype(int) == fold,
        0
    ].astype(str)

    train_idx = np.isin(session, train_sessions)
    test_idx = np.isin(session, test_sessions)

    X_train = X_obs[train_idx]
    y_train = y[train_idx]

    X_test = X_obs[test_idx]
    y_test = y[test_idx]

    print("Training trials:", X_train.shape[0])
    print("Test trials:", X_test.shape[0])
    print("Training sessions:", len(train_sessions))
    print("Test sessions:", len(test_sessions))

    w_hat, se, train_ll, result = fit_logistic_glm(
        X_train,
        y_train,
        prior_sigma=prior_sigma
    )

    test_ll, test_acc = evaluate_logistic_glm(w_hat, X_test, y_test)
    train_test_ll_per_trial = test_ll / len(y_test)

    print("Converged:", result.success)
    print("Train LL:", train_ll)
    print("Test LL:", test_ll)
    print("Test LL per trial:", train_test_ll_per_trial)
    print("Test accuracy:", test_acc)
    print("Weights:", w_hat)


Fitting 1-state GLM pre-training model: fold 0


NameError: name 'X_obs' is not defined

In [20]:
# ============================================================
# Step 3. Pre-training: fit 1-state GLM for choice prediction
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

# ------------------------------------------------------------
# 1) Load model-ready arrays
# ------------------------------------------------------------

DATA_DIR = Path("/content/wm_glmhmm_data")
RESULTS_DIR = Path("/content/wm_glmhmm_results")

combined_npz_path = DATA_DIR / "combined_all_mice.npz"
fold_npz_path = DATA_DIR / "fold_mapping_session_all_mice.npz"

container = np.load(combined_npz_path, allow_pickle=True)

obs_mat = container["arr_0"]
trans_mat = container["arr_1"]
output = container["arr_2"].astype(int)
session = container["arr_3"].astype(str)

fold_container = np.load(fold_npz_path, allow_pickle=True)
fold_mapping_session = fold_container["arr_0"]

print("obs_mat shape:", obs_mat.shape)
print("trans_mat shape:", trans_mat.shape)
print("output shape:", output.shape)
print("session shape:", session.shape)
print("fold_mapping_session shape:", fold_mapping_session.shape)

# ------------------------------------------------------------
# 2) Prepare observation matrix
# ------------------------------------------------------------

X_obs = obs_mat.copy()
X_obs[:, -1] = 1.0

y = output[:, 0].astype(int)

assert set(np.unique(y)).issubset({0, 1})
assert np.isfinite(X_obs).all()

print("Prepared X_obs shape:", X_obs.shape)
print("Choice counts:")
print(pd.Series(y).value_counts().sort_index())

# ------------------------------------------------------------
# 3) Define Bernoulli logistic regression functions
# ------------------------------------------------------------

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def neg_log_likelihood_and_grad(w, X, y, prior_sigma=100.0):
    logits = X @ w

    nll = np.sum(np.logaddexp(0, logits) - y * logits)
    nll += 0.5 * np.sum((w / prior_sigma) ** 2)

    p = sigmoid(logits)
    grad = X.T @ (p - y)
    grad += w / (prior_sigma ** 2)

    return nll, grad


def fit_logistic_glm(X_train, y_train, prior_sigma=100.0):
    w0 = np.zeros(X_train.shape[1])

    result = minimize(
        fun=lambda w: neg_log_likelihood_and_grad(w, X_train, y_train, prior_sigma)[0],
        x0=w0,
        jac=lambda w: neg_log_likelihood_and_grad(w, X_train, y_train, prior_sigma)[1],
        method="BFGS",
        options={"maxiter": 1000, "gtol": 1e-6}
    )

    if not result.success:
        print("Warning: GLM optimization did not fully converge.")
        print(result.message)

    w_hat = result.x

    try:
        cov = np.asarray(result.hess_inv)
        se = np.sqrt(np.diag(cov))
    except Exception:
        se = np.full_like(w_hat, np.nan)

    train_nll, _ = neg_log_likelihood_and_grad(w_hat, X_train, y_train, prior_sigma)
    train_ll = -train_nll

    return w_hat, se, train_ll, result


def evaluate_logistic_glm(w, X_test, y_test):
    logits = X_test @ w
    p_right = sigmoid(logits)

    test_ll = -np.sum(np.logaddexp(0, logits) - y_test * logits)

    y_pred = (p_right >= 0.5).astype(int)
    acc = np.mean(y_pred == y_test)

    return test_ll, acc

# ------------------------------------------------------------
# 4) Fit 1-state GLM separately for each held-out fold
# ------------------------------------------------------------

N_FOLDS = 5
prior_sigma = 100.0

glm_summary_rows = []

for fold in range(N_FOLDS):

    print("\n================================================")
    print(f"Fitting 1-state GLM pre-training model: fold {fold}")
    print("================================================")

    train_sessions = fold_mapping_session[
        fold_mapping_session[:, 1].astype(int) != fold,
        0
    ].astype(str)

    test_sessions = fold_mapping_session[
        fold_mapping_session[:, 1].astype(int) == fold,
        0
    ].astype(str)

    train_idx = np.isin(session, train_sessions)
    test_idx = np.isin(session, test_sessions)

    X_train = X_obs[train_idx]
    y_train = y[train_idx]

    X_test = X_obs[test_idx]
    y_test = y[test_idx]

    print("Training trials:", X_train.shape[0])
    print("Test trials:", X_test.shape[0])
    print("Training sessions:", len(train_sessions))
    print("Test sessions:", len(test_sessions))

    w_hat, se, train_ll, result = fit_logistic_glm(
        X_train,
        y_train,
        prior_sigma=prior_sigma
    )

    test_ll, test_acc = evaluate_logistic_glm(w_hat, X_test, y_test)
    test_ll_per_trial = test_ll / len(y_test)

    print("Converged:", result.success)
    print("Train LL:", train_ll)
    print("Test LL:", test_ll)
    print("Test LL per trial:", test_ll_per_trial)
    print("Test accuracy:", test_acc)
    print("Weights:", w_hat)

    # --------------------------------------------------------
    # Save in the same location/name pattern that the author's
    # GLM-HMM script expects.
    # --------------------------------------------------------

    fold_dir = RESULTS_DIR / "Model" / "glm_#state=1" / f"fld_num={fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    save_path = fold_dir / "important_params_iter_0.npz"

    recovered_weights = np.zeros((1, 2, X_obs.shape[1]))
    recovered_weights[0, 1, :] = w_hat

    standard_deviation = np.zeros((1, 2, X_obs.shape[1]))
    standard_deviation[0, 1, :] = se

    train_calculate_LL = np.array([train_ll])

    np.savez(
        save_path,
        train_calculate_LL,
        recovered_weights,
        standard_deviation
    )

    glm_summary_rows.append({
        "fold": fold,
        "n_train_trials": X_train.shape[0],
        "n_test_trials": X_test.shape[0],
        "n_train_sessions": len(train_sessions),
        "n_test_sessions": len(test_sessions),
        "train_ll": train_ll,
        "test_ll": test_ll,
        "test_ll_per_trial": test_ll_per_trial,
        "test_accuracy": test_acc,
        "converged": result.success,
        "save_path": str(save_path)
    })

# ------------------------------------------------------------
# 5) Save and display GLM pre-training summary
# ------------------------------------------------------------

glm_summary = pd.DataFrame(glm_summary_rows)

summary_path = RESULTS_DIR / "glm_pretraining_summary.csv"
glm_summary.to_csv(summary_path, index=False)

print("\nGLM pre-training summary:")
display(glm_summary)

print("\nSaved summary to:")
print(summary_path)

print("\nSaved fold-specific initialization files under:")
print(RESULTS_DIR / "Model" / "glm_#state=1")

obs_mat shape: (12150, 5)
trans_mat shape: (12150, 6)
output shape: (12150, 1)
session shape: (12150,)
fold_mapping_session shape: (405, 2)
Prepared X_obs shape: (12150, 5)
Choice counts:
0    6288
1    5862
Name: count, dtype: int64

Fitting 1-state GLM pre-training model: fold 0
Training trials: 9450
Test trials: 2700
Training sessions: 315
Test sessions: 90
Converged: True
Train LL: -5884.439586048808
Test LL: -1676.0319517477114
Test LL per trial: -0.6207525747213746
Test accuracy: 0.6877777777777778
Weights: [ 0.77752966  0.05583328 -0.02058973  0.00488999 -0.07806139]

Fitting 1-state GLM pre-training model: fold 1
Training trials: 9450
Test trials: 2700
Training sessions: 315
Test sessions: 90
Converged: True
Train LL: -5872.451813100179
Test LL: -1688.5501410551092
Test LL per trial: -0.6253889411315219
Test accuracy: 0.6803703703703704
Weights: [ 0.78692801  0.04971414 -0.03611706 -0.00646213 -0.05018866]

Fitting 1-state GLM pre-training model: fold 2
Training trials: 9450
Te

,fold,n_train_trials,n_test_trials,n_train_sessions,n_test_sessions,train_ll,test_ll,test_ll_per_trial,test_accuracy,converged,save_path
0,0,9450,2700,315,90,-5884.439586,-1676.031952,-0.620753,0.687778,True,/content/wm_glmhmm_results/Model/glm_#state=1/...
1,1,9450,2700,315,90,-5872.451813,-1688.550141,-0.625389,0.680370,True,/content/wm_glmhmm_results/Model/glm_#state=1/...
2,2,9450,2700,315,90,-5853.848923,-1707.377939,-0.632362,0.671481,True,/content/wm_glmhmm_results/Model/glm_#state=1/...
3,3,9449,2701,315,90,-5888.727413,-1672.568518,-0.619240,0.690115,False,/content/wm_glmhmm_results/Model/glm_#state=1/...
4,4,10801,1349,360,45,-6739.069333,-821.763708,-0.609165,0.703484,False,/content/wm_glmhmm_results/Model/glm_#state=1/...



Saved summary to:
/content/wm_glmhmm_results/glm_pretraining_summary.csv

Saved fold-specific initialization files under:
/content/wm_glmhmm_results/Model/glm_#state=1


In [21]:
# ============================================================
# Step 3b. Stabilize 1-state GLM pre-training with L-BFGS-B
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

DATA_DIR = Path("/content/wm_glmhmm_data")
RESULTS_DIR = Path("/content/wm_glmhmm_results")

combined_npz_path = DATA_DIR / "combined_all_mice.npz"
fold_npz_path = DATA_DIR / "fold_mapping_session_all_mice.npz"

container = np.load(combined_npz_path, allow_pickle=True)

obs_mat = container["arr_0"]
trans_mat = container["arr_1"]
output = container["arr_2"].astype(int)
session = container["arr_3"].astype(str)

fold_container = np.load(fold_npz_path, allow_pickle=True)
fold_mapping_session = fold_container["arr_0"]

X_obs = obs_mat.copy()
X_obs[:, -1] = 1.0

y = output[:, 0].astype(int)

print("Loaded data:")
print("X_obs:", X_obs.shape)
print("y:", y.shape)
print("session:", session.shape)
print("fold mapping:", fold_mapping_session.shape)


# ------------------------------------------------------------
# Logistic GLM helper functions
# ------------------------------------------------------------

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def neg_log_likelihood_and_grad(w, X, y, prior_sigma=100.0):
    logits = X @ w

    nll = np.sum(np.logaddexp(0, logits) - y * logits)
    nll += 0.5 * np.sum((w / prior_sigma) ** 2)

    p = sigmoid(logits)

    grad = X.T @ (p - y)
    grad += w / (prior_sigma ** 2)

    return nll, grad


def fit_logistic_glm_lbfgsb(X_train, y_train, prior_sigma=100.0):
    w0 = np.zeros(X_train.shape[1])

    result = minimize(
        fun=lambda w: neg_log_likelihood_and_grad(w, X_train, y_train, prior_sigma)[0],
        x0=w0,
        jac=lambda w: neg_log_likelihood_and_grad(w, X_train, y_train, prior_sigma)[1],
        method="L-BFGS-B",
        options={
            "maxiter": 2000,
            "ftol": 1e-10,
            "gtol": 1e-6,
            "maxls": 50
        }
    )

    w_hat = result.x

    train_nll, grad = neg_log_likelihood_and_grad(
        w_hat,
        X_train,
        y_train,
        prior_sigma
    )

    train_ll = -train_nll
    grad_norm = np.linalg.norm(grad)

    # L-BFGS-B does not directly return a full Hessian inverse like BFGS.
    # For initialization, exact SE is not essential, so we save NaN SE.
    se = np.full_like(w_hat, np.nan)

    return w_hat, se, train_ll, grad_norm, result


def evaluate_logistic_glm(w, X_test, y_test):
    logits = X_test @ w

    test_ll = -np.sum(np.logaddexp(0, logits) - y_test * logits)

    p_right = sigmoid(logits)
    y_pred = (p_right >= 0.5).astype(int)

    accuracy = np.mean(y_pred == y_test)

    return test_ll, accuracy


# ------------------------------------------------------------
# Refit all folds using L-BFGS-B and overwrite initialization files
# ------------------------------------------------------------

N_FOLDS = 5
prior_sigma = 100.0

glm_refit_rows = []

for fold in range(N_FOLDS):

    print("\n================================================")
    print(f"Refitting 1-state GLM pre-training model: fold {fold}")
    print("================================================")

    train_sessions = fold_mapping_session[
        fold_mapping_session[:, 1].astype(int) != fold,
        0
    ].astype(str)

    test_sessions = fold_mapping_session[
        fold_mapping_session[:, 1].astype(int) == fold,
        0
    ].astype(str)

    train_idx = np.isin(session, train_sessions)
    test_idx = np.isin(session, test_sessions)

    X_train = X_obs[train_idx]
    y_train = y[train_idx]

    X_test = X_obs[test_idx]
    y_test = y[test_idx]

    w_hat, se, train_ll, grad_norm, result = fit_logistic_glm_lbfgsb(
        X_train,
        y_train,
        prior_sigma=prior_sigma
    )

    test_ll, test_accuracy = evaluate_logistic_glm(
        w_hat,
        X_test,
        y_test
    )

    test_ll_per_trial = test_ll / len(y_test)

    print("Converged:", result.success)
    print("Message:", result.message)
    print("Gradient norm:", grad_norm)
    print("Train LL:", train_ll)
    print("Test LL:", test_ll)
    print("Test LL per trial:", test_ll_per_trial)
    print("Test accuracy:", test_accuracy)
    print("Weights:", w_hat)

    # Save in the same expected initialization structure
    fold_dir = RESULTS_DIR / "Model" / "glm_#state=1" / f"fld_num={fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    save_path = fold_dir / "important_params_iter_0.npz"

    recovered_weights = np.zeros((1, 2, X_obs.shape[1]))
    recovered_weights[0, 1, :] = w_hat

    standard_deviation = np.zeros((1, 2, X_obs.shape[1]))
    standard_deviation[0, 1, :] = se

    train_calculate_LL = np.array([train_ll])

    np.savez(
        save_path,
        train_calculate_LL,
        recovered_weights,
        standard_deviation
    )

    glm_refit_rows.append({
        "fold": fold,
        "n_train_trials": X_train.shape[0],
        "n_test_trials": X_test.shape[0],
        "n_train_sessions": len(train_sessions),
        "n_test_sessions": len(test_sessions),
        "converged": result.success,
        "message": str(result.message),
        "gradient_norm": grad_norm,
        "train_ll": train_ll,
        "test_ll": test_ll,
        "test_ll_per_trial": test_ll_per_trial,
        "test_accuracy": test_accuracy,
        "w_correct_side": w_hat[0],
        "w_prev_choice": w_hat[1],
        "w_prev_reward": w_hat[2],
        "w_delay": w_hat[3],
        "w_bias": w_hat[4],
        "save_path": str(save_path)
    })


# ------------------------------------------------------------
# Save stabilized GLM pre-training summary
# ------------------------------------------------------------

glm_refit_summary = pd.DataFrame(glm_refit_rows)

summary_path = RESULTS_DIR / "glm_pretraining_summary_LBFGSB.csv"
glm_refit_summary.to_csv(summary_path, index=False)

print("\nStabilized GLM pre-training summary:")
display(glm_refit_summary)

print("\nSaved stabilized summary to:")
print(summary_path)

print("\nUpdated initialization files are saved under:")
print(RESULTS_DIR / "Model" / "glm_#state=1")

Loaded data:
X_obs: (12150, 5)
y: (12150,)
session: (12150,)
fold mapping: (405, 2)

Refitting 1-state GLM pre-training model: fold 0
Converged: True
Message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Gradient norm: 0.0001020887448499943
Train LL: -5884.439586048812
Test LL: -1676.031950928134
Test LL per trial: -0.6207525744178274
Test accuracy: 0.6877777777777778
Weights: [ 0.77752966  0.05583324 -0.02058977  0.00489    -0.07806137]

Refitting 1-state GLM pre-training model: fold 1
Converged: True
Message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Gradient norm: 0.0005639331741813336
Train LL: -5872.451813100259
Test LL: -1688.550138370058
Test LL per trial: -0.6253889401370585
Test accuracy: 0.6803703703703704
Weights: [ 0.78692799  0.04971416 -0.03611704 -0.00646187 -0.05018856]

Refitting 1-state GLM pre-training model: fold 2
Converged: True
Message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Gradient norm: 0.00029311523802279677
Train LL: -5853.84

,fold,n_train_trials,n_test_trials,n_train_sessions,n_test_sessions,converged,message,gradient_norm,train_ll,test_ll,test_ll_per_trial,test_accuracy,w_correct_side,w_prev_choice,w_prev_reward,w_delay,w_bias,save_path
0,0,9450,2700,315,90,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,0.000102,-5884.439586,-1676.031951,-0.620753,0.687778,0.777530,0.055833,-0.020590,0.004890,-0.078061,/content/wm_glmhmm_results/Model/glm_#state=1/...
1,1,9450,2700,315,90,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,0.000564,-5872.451813,-1688.550138,-0.625389,0.680370,0.786928,0.049714,-0.036117,-0.006462,-0.050189,/content/wm_glmhmm_results/Model/glm_#state=1/...
2,2,9450,2700,315,90,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,0.000293,-5853.848923,-1707.377947,-0.632362,0.671481,0.797862,0.037712,-0.015855,0.019362,-0.065657,/content/wm_glmhmm_results/Model/glm_#state=1/...
3,3,9449,2701,315,90,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,0.000213,-5888.727413,-1672.568521,-0.619240,0.690115,0.773030,0.030116,-0.012984,0.008110,-0.093405,/content/wm_glmhmm_results/Model/glm_#state=1/...
4,4,10801,1349,360,45,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,0.000979,-6739.069333,-821.763706,-0.609165,0.703484,0.771932,0.063090,-0.019076,-0.003841,-0.068348,/content/wm_glmhmm_results/Model/glm_#state=1/...



Saved stabilized summary to:
/content/wm_glmhmm_results/glm_pretraining_summary_LBFGSB.csv

Updated initialization files are saved under:
/content/wm_glmhmm_results/Model/glm_#state=1


In [22]:
# ============================================================
# Step 4 fixed. Short test run: fit K=2 GLM-HMM on fold 0
# ============================================================

import os
import sys
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import ssm

# ------------------------------------------------------------
# 1) Load GLM-HMM-ready data
# ------------------------------------------------------------

DATA_DIR = Path("/content/wm_glmhmm_data")
RESULTS_DIR = Path("/content/wm_glmhmm_results")

combined_npz_path = DATA_DIR / "combined_all_mice.npz"
fold_npz_path = DATA_DIR / "fold_mapping_session_all_mice.npz"

container = np.load(combined_npz_path, allow_pickle=True)

obs_mat = container["arr_0"]
trans_mat = container["arr_1"]
output = container["arr_2"].astype(int)
session = container["arr_3"].astype(str)

fold_container = np.load(fold_npz_path, allow_pickle=True)
fold_mapping_session = fold_container["arr_0"]

# Match the author's logic: final observation column is intercept
obs_mat = obs_mat.copy()
obs_mat[:, -1] = 1.0

# No violation trials in this data
mask = np.ones(output.shape[0], dtype=bool)

print("obs_mat:", obs_mat.shape)
print("trans_mat:", trans_mat.shape)
print("output:", output.shape)
print("session:", session.shape)
print("mask:", mask.shape)


# ------------------------------------------------------------
# 2) Segment arrays by session
# ------------------------------------------------------------

def segment_by_session(obs_mat, trans_mat, y, mask, session):
    """
    Split trial-level arrays into session-level sequences.
    Each mouse/session/delay block becomes one HMM sequence.
    """

    inputs_obs = []
    inputs_trans = []
    datas = []
    masks = []

    unique_sessions = pd.Series(session).drop_duplicates().to_numpy()

    counter = 0

    for sess in unique_sessions:
        idx = np.where(session == sess)[0]

        inputs_obs.append(obs_mat[idx, :])
        inputs_trans.append(trans_mat[idx, :])
        datas.append(y[idx, :])
        masks.append(mask[idx])

        counter += len(idx)

    assert counter == obs_mat.shape[0], "Some trials were not assigned to sessions."

    return inputs_obs, inputs_trans, datas, masks


# ------------------------------------------------------------
# 3) Model settings for short test
# ------------------------------------------------------------

K = 2
D = 1
C = 2
fold = 0
iter_id = 0

M_obs = obs_mat.shape[1]
M_trans = trans_mat.shape[1]

num_iters_EM_fit = 25
prior_sigma = 100.0
transition_alpha = 1.0
GLM_T_init_0 = True

print("K:", K)
print("fold:", fold)
print("M_obs:", M_obs)
print("M_trans:", M_trans)


# ------------------------------------------------------------
# 4) Split training and held-out sessions
# ------------------------------------------------------------

train_sessions = fold_mapping_session[
    fold_mapping_session[:, 1].astype(int) != fold,
    0
].astype(str)

test_sessions = fold_mapping_session[
    fold_mapping_session[:, 1].astype(int) == fold,
    0
].astype(str)

train_idx = np.isin(session, train_sessions)
test_idx = np.isin(session, test_sessions)

obs_train = obs_mat[train_idx, :]
trans_train = trans_mat[train_idx, :]
y_train = output[train_idx, :]
mask_train = mask[train_idx]
session_train = session[train_idx]

obs_test = obs_mat[test_idx, :]
trans_test = trans_mat[test_idx, :]
y_test = output[test_idx, :]
mask_test = mask[test_idx]
session_test = session[test_idx]

inputs_train, inputs_trans_train, datas_train, masks_train = segment_by_session(
    obs_train,
    trans_train,
    y_train,
    mask_train,
    session_train
)

inputs_test, inputs_trans_test, datas_test, masks_test = segment_by_session(
    obs_test,
    trans_test,
    y_test,
    mask_test,
    session_test
)

print("Training trials:", y_train.shape[0])
print("Held-out test trials:", y_test.shape[0])
print("Training sessions:", len(np.unique(session_train)))
print("Held-out test sessions:", len(np.unique(session_test)))
print("Number of training sequences:", len(datas_train))
print("Number of held-out test sequences:", len(datas_test))
print("Example training sequence shape:", datas_train[0].shape)


# ------------------------------------------------------------
# 5) Load 1-state GLM initialization
# ------------------------------------------------------------

glm_init_path = (
    RESULTS_DIR
    / "Model"
    / "glm_#state=1"
    / f"fld_num={fold}"
    / "important_params_iter_0.npz"
)

if not glm_init_path.exists():
    raise FileNotFoundError(f"Could not find GLM initialization file: {glm_init_path}")

glm_container = np.load(glm_init_path, allow_pickle=True)

glm_train_ll = glm_container["arr_0"]
glm_weights_1state = glm_container["arr_1"]

print("Loaded GLM initialization from:")
print(glm_init_path)
print("Original saved GLM weights shape:", glm_weights_1state.shape)

# ------------------------------------------------------------
# Important correction:
# The custom input_driven_obs_diff_inputs observation model expects
# K x (C - 1) x M_obs parameters.
#
# Since C = 2, it expects K x 1 x M_obs.
# Our saved GLM file contains class 0 baseline + class 1 right-choice weights:
# 1 x 2 x M_obs.
# Therefore, extract only class 1.
# ------------------------------------------------------------

if glm_weights_1state.shape[1] == 2:
    glm_weights_for_hmm = glm_weights_1state[:, 1:2, :]
elif glm_weights_1state.shape[1] == 1:
    glm_weights_for_hmm = glm_weights_1state
else:
    raise ValueError("Unexpected GLM weight shape.")

print("Corrected GLM weights for HMM shape:", glm_weights_for_hmm.shape)


# ------------------------------------------------------------
# 6) Instantiate GLM-HMM using author's model structure
# ------------------------------------------------------------

this_hmm = ssm.HMM_TO(
    K,
    D,
    M_trans=M_trans,
    M_obs=M_obs,
    observations="input_driven_obs_diff_inputs",
    observation_kwargs=dict(
        C=C,
        prior_sigma=prior_sigma
    ),
    transitions="inputdrivenalt",
    transition_kwargs=dict(
        prior_sigma=prior_sigma,
        alpha=transition_alpha,
        kappa=0
    )
)

print("Created HMM_TO model.")
print("Default observation params shape:", this_hmm.observations.params.shape)


# ------------------------------------------------------------
# 7) Initialize observation GLM weights correctly
# ------------------------------------------------------------

rng = np.random.default_rng(iter_id)

obs_init = np.tile(glm_weights_for_hmm, (K, 1, 1))
obs_init = obs_init + rng.normal(0, 0.05, size=obs_init.shape)

# This should be K x 1 x M_obs
assert obs_init.shape == (K, C - 1, M_obs)

this_hmm.observations.params = obs_init

print("Initialized observation weights.")
print("Observation params shape after init:", this_hmm.observations.params.shape)


# ------------------------------------------------------------
# 8) Initialize transition GLM weights at zero
# ------------------------------------------------------------

if GLM_T_init_0:
    try:
        print("Default transition params structure:")
        print(type(this_hmm.transitions.params))
        print("Transition params[1] shape:", this_hmm.transitions.params[1].shape)

        this_hmm.transitions.params[1][...] = np.zeros_like(
            this_hmm.transitions.params[1]
        )

        print("Transition GLM weights initialized to zero.")
    except Exception as e:
        print("Could not manually zero transition weights.")
        print("Reason:", repr(e))
        print("Continuing with default transition initialization.")


# ------------------------------------------------------------
# 9) Fit GLM-HMM with EM on training sessions
# ------------------------------------------------------------

print("\n=== Fitting K=2 GLM-HMM short test run ===")

lls = this_hmm.fit(
    datas_train,
    transition_input=inputs_trans_train,
    observation_input=inputs_train,
    train_masks=masks_train,
    method="em",
    num_iters=num_iters_EM_fit,
    initialize=False,
    tolerance=1e-4
)

print("Finished short test fit.")
print("Number of LL values:", len(lls))
print("Initial train LL:", lls[0])
print("Final train LL:", lls[-1])
print("LL improvement:", lls[-1] - lls[0])


# ------------------------------------------------------------
# 10) Compute held-out test log-likelihood
# ------------------------------------------------------------

def compute_glmhmm_log_probability(model, datas, obs_inputs, trans_inputs, masks):
    try:
        return model.log_probability(
            datas,
            transition_input=trans_inputs,
            observation_input=obs_inputs,
            masks=masks
        )
    except TypeError:
        pass

    try:
        return model.log_probability(
            datas,
            transition_input=trans_inputs,
            observation_input=obs_inputs
        )
    except TypeError:
        pass

    raise TypeError(
        "The model fit worked, but this ssm fork uses a different log_probability API."
    )


test_ll = compute_glmhmm_log_probability(
    this_hmm,
    datas_test,
    inputs_test,
    inputs_trans_test,
    masks_test
)

test_n_trials = y_test.shape[0]
test_ll_per_trial = test_ll / test_n_trials

p_right_baseline = np.mean(y_train[:, 0])

baseline_test_ll = np.sum(
    y_test[:, 0] * np.log(p_right_baseline)
    + (1 - y_test[:, 0]) * np.log(1 - p_right_baseline)
)

bits_per_trial = (test_ll - baseline_test_ll) / (
    test_n_trials * np.log(2)
)

print("\nHeld-out evaluation:")
print("Test LL:", test_ll)
print("Test LL per trial:", test_ll_per_trial)
print("Baseline test LL:", baseline_test_ll)
print("Bits per trial over Bernoulli baseline:", bits_per_trial)


# ------------------------------------------------------------
# 11) Save short test result
# ------------------------------------------------------------

short_test_dir = (
    RESULTS_DIR
    / "Model"
    / f"glmhmm_#state={K}"
    / f"fld_num={fold}"
    / f"iter_{iter_id}"
)

short_test_dir.mkdir(parents=True, exist_ok=True)

raw_param_path = short_test_dir / f"glm_hmm_raw_parameters_itr_{iter_id}.npz"
summary_path = short_test_dir / f"glmhmm_short_test_summary_K{K}_fold{fold}.csv"

np.savez(
    raw_param_path,
    this_hmm.params,
    lls
)

short_summary = pd.DataFrame([{
    "K": K,
    "fold": fold,
    "iter": iter_id,
    "num_iters_EM_fit": num_iters_EM_fit,
    "n_train_trials": y_train.shape[0],
    "n_test_trials": y_test.shape[0],
    "n_train_sessions": len(np.unique(session_train)),
    "n_test_sessions": len(np.unique(session_test)),
    "initial_train_ll": lls[0],
    "final_train_ll": lls[-1],
    "ll_improvement": lls[-1] - lls[0],
    "test_ll": test_ll,
    "test_ll_per_trial": test_ll_per_trial,
    "baseline_test_ll": baseline_test_ll,
    "bits_per_trial_over_baseline": bits_per_trial,
    "raw_param_path": str(raw_param_path)
}])

short_summary.to_csv(summary_path, index=False)

print("\nSaved raw GLM-HMM parameters to:")
print(raw_param_path)

print("\nSaved short test summary to:")
print(summary_path)

display(short_summary)

obs_mat: (12150, 5)
trans_mat: (12150, 6)
output: (12150, 1)
session: (12150,)
mask: (12150,)
K: 2
fold: 0
M_obs: 5
M_trans: 6
Training trials: 9450
Held-out test trials: 2700
Training sessions: 315
Held-out test sessions: 90
Number of training sequences: 315
Number of held-out test sequences: 90
Example training sequence shape: (30, 1)
Loaded GLM initialization from:
/content/wm_glmhmm_results/Model/glm_#state=1/fld_num=0/important_params_iter_0.npz
Original saved GLM weights shape: (1, 2, 5)
Corrected GLM weights for HMM shape: (1, 1, 5)
Created HMM_TO model.
Default observation params shape: (2, 1, 5)
Initialized observation weights.
Observation params shape after init: (2, 1, 5)
Default transition params structure:
<class 'list'>
Transition params[1] shape: (1, 6)
Transition GLM weights initialized to zero.

=== Fitting K=2 GLM-HMM short test run ===


  0%|          | 0/25 [00:00<?, ?it/s]

Finished short test fit.
Number of LL values: 26
Initial train LL: -8717.120670313812
Final train LL: -5801.8011563565615
LL improvement: 2915.319513957251


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [23]:
# ============================================================
# Step 4b. Fix held-out likelihood evaluation after successful fitting
# Do NOT refit the model. This uses the fitted this_hmm object.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("/content/wm_glmhmm_results")

# ------------------------------------------------------------
# 1) Check that the fitted model and test data exist
# ------------------------------------------------------------

required_vars = [
    "this_hmm",
    "lls",
    "datas_test",
    "inputs_test",
    "inputs_trans_test",
    "masks_test",
    "y_train",
    "y_test",
    "session_train",
    "session_test",
    "K",
    "fold",
    "iter_id",
    "num_iters_EM_fit"
]

missing_vars = [v for v in required_vars if v not in globals()]

if len(missing_vars) > 0:
    raise NameError(
        "Some required variables are missing. "
        "You may need to rerun the fixed Step 4 fitting cell first. "
        f"Missing variables: {missing_vars}"
    )

print("Fitted GLM-HMM object found.")
print("Final training LL:", lls[-1])


# ------------------------------------------------------------
# 2) Make sure data and masks are all 2D: T x 1
# ------------------------------------------------------------

datas_test_2d = [
    np.asarray(d).reshape(-1, 1).astype(int)
    for d in datas_test
]

masks_test_2d = [
    np.asarray(m).reshape(-1, 1).astype(bool)
    for m in masks_test
]

inputs_test_2d = [
    np.asarray(x)
    for x in inputs_test
]

inputs_trans_test_2d = [
    np.asarray(x)
    for x in inputs_trans_test
]

print("Example data shape:", datas_test_2d[0].shape)
print("Example mask shape:", masks_test_2d[0].shape)
print("Example obs input shape:", inputs_test_2d[0].shape)
print("Example transition input shape:", inputs_trans_test_2d[0].shape)


# ------------------------------------------------------------
# 3) Compute held-out log-likelihood safely
# ------------------------------------------------------------

def safe_glmhmm_log_probability(model, datas, obs_inputs, trans_inputs, masks):
    """
    Compute log probability for a list of held-out sessions.
    This custom ssm fork expects data and mask to be 2D: T x 1.
    """

    # First try the list version
    try:
        lp = model.log_probability(
            datas,
            transition_input=trans_inputs,
            observation_input=obs_inputs,
            masks=masks
        )
        return float(np.asarray(lp).sum())
    except Exception as e:
        print("List-level log_probability failed.")
        print("Reason:", repr(e))
        print("Trying sequence-by-sequence evaluation...")

    # If list version fails, evaluate each session separately
    total_lp = 0.0

    for i in range(len(datas)):
        d = datas[i]
        oi = obs_inputs[i]
        ti = trans_inputs[i]
        m = masks[i]

        try:
            lp_i = model.log_probability(
                d,
                transition_input=ti,
                observation_input=oi,
                masks=m
            )
        except TypeError:
            lp_i = model.log_probability(
                [d],
                transition_input=[ti],
                observation_input=[oi],
                masks=[m]
            )

        total_lp += float(np.asarray(lp_i).sum())

    return total_lp


test_ll = safe_glmhmm_log_probability(
    this_hmm,
    datas_test_2d,
    inputs_test_2d,
    inputs_trans_test_2d,
    masks_test_2d
)

test_n_trials = int(sum(d.shape[0] for d in datas_test_2d))
test_ll_per_trial = test_ll / test_n_trials

print("\nHeld-out GLM-HMM evaluation:")
print("Test LL:", test_ll)
print("Test trials:", test_n_trials)
print("Test LL per trial:", test_ll_per_trial)


# ------------------------------------------------------------
# 4) Compute Bernoulli baseline likelihood
# ------------------------------------------------------------

p_right_baseline = np.mean(y_train[:, 0])

# Avoid log(0) just in case
eps = 1e-12
p_right_baseline = np.clip(p_right_baseline, eps, 1 - eps)

baseline_test_ll = np.sum(
    y_test[:, 0] * np.log(p_right_baseline)
    + (1 - y_test[:, 0]) * np.log(1 - p_right_baseline)
)

bits_per_trial = (test_ll - baseline_test_ll) / (
    test_n_trials * np.log(2)
)

print("\nBaseline comparison:")
print("Training-set baseline p(right):", p_right_baseline)
print("Baseline test LL:", baseline_test_ll)
print("Bits per trial over Bernoulli baseline:", bits_per_trial)


# ------------------------------------------------------------
# 5) Save short test results
# ------------------------------------------------------------

short_test_dir = (
    RESULTS_DIR
    / "Model"
    / f"glmhmm_#state={K}"
    / f"fld_num={fold}"
    / f"iter_{iter_id}"
)

short_test_dir.mkdir(parents=True, exist_ok=True)

raw_param_path = short_test_dir / f"glm_hmm_raw_parameters_itr_{iter_id}.npz"
summary_path = short_test_dir / f"glmhmm_short_test_summary_K{K}_fold{fold}.csv"

np.savez(
    raw_param_path,
    params=np.array(this_hmm.params, dtype=object),
    train_lls=np.asarray(lls)
)

short_summary = pd.DataFrame([{
    "K": K,
    "fold": fold,
    "iter": iter_id,
    "num_iters_EM_fit": num_iters_EM_fit,
    "n_train_trials": y_train.shape[0],
    "n_test_trials": test_n_trials,
    "n_train_sessions": len(np.unique(session_train)),
    "n_test_sessions": len(np.unique(session_test)),
    "initial_train_ll": float(lls[0]),
    "final_train_ll": float(lls[-1]),
    "ll_improvement": float(lls[-1] - lls[0]),
    "test_ll": test_ll,
    "test_ll_per_trial": test_ll_per_trial,
    "baseline_test_ll": baseline_test_ll,
    "bits_per_trial_over_baseline": bits_per_trial,
    "raw_param_path": str(raw_param_path)
}])

short_summary.to_csv(summary_path, index=False)

print("\nSaved raw GLM-HMM parameters to:")
print(raw_param_path)

print("\nSaved short test summary to:")
print(summary_path)

display(short_summary)

Fitted GLM-HMM object found.
Final training LL: -5801.8011563565615
Example data shape: (30, 1)
Example mask shape: (30, 1)
Example obs input shape: (30, 5)
Example transition input shape: (30, 6)

Held-out GLM-HMM evaluation:
Test LL: -1694.767616619928
Test trials: 2700
Test LL per trial: -0.6276917098592326

Baseline comparison:
Training-set baseline p(right): 0.48074074074074075
Baseline test LL: -1871.1125075402192
Bits per trial over Bernoulli baseline: 0.09422662948770713

Saved raw GLM-HMM parameters to:
/content/wm_glmhmm_results/Model/glmhmm_#state=2/fld_num=0/iter_0/glm_hmm_raw_parameters_itr_0.npz

Saved short test summary to:
/content/wm_glmhmm_results/Model/glmhmm_#state=2/fld_num=0/iter_0/glmhmm_short_test_summary_K2_fold0.csv


,K,fold,iter,num_iters_EM_fit,n_train_trials,n_test_trials,n_train_sessions,n_test_sessions,initial_train_ll,final_train_ll,ll_improvement,test_ll,test_ll_per_trial,baseline_test_ll,bits_per_trial_over_baseline,raw_param_path
0,2,0,0,25,9450,2700,315,90,-8717.12067,-5801.801156,2915.319514,-1694.767617,-0.627692,-1871.112508,0.094227,/content/wm_glmhmm_results/Model/glmhmm_#state...


In [24]:
# ============================================================
# Step 5. Full GLM-HMM run for K=2, fold 0, multiple initializations
# ============================================================

import os
import sys
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import ssm

# ------------------------------------------------------------
# 1) Load model-ready arrays
# ------------------------------------------------------------

DATA_DIR = Path("/content/wm_glmhmm_data")
RESULTS_DIR = Path("/content/wm_glmhmm_results")

combined_npz_path = DATA_DIR / "combined_all_mice.npz"
fold_npz_path = DATA_DIR / "fold_mapping_session_all_mice.npz"

container = np.load(combined_npz_path, allow_pickle=True)

obs_mat = container["arr_0"]
trans_mat = container["arr_1"]
output = container["arr_2"].astype(int)
session = container["arr_3"].astype(str)

fold_container = np.load(fold_npz_path, allow_pickle=True)
fold_mapping_session = fold_container["arr_0"]

# Match author's logic: final observation column is intercept
obs_mat = obs_mat.copy()
obs_mat[:, -1] = 1.0

mask = np.ones(output.shape[0], dtype=bool)

print("obs_mat:", obs_mat.shape)
print("trans_mat:", trans_mat.shape)
print("output:", output.shape)
print("session:", session.shape)
print("fold mapping:", fold_mapping_session.shape)


# ------------------------------------------------------------
# 2) Helper: segment arrays by session
# ------------------------------------------------------------

def segment_by_session(obs_mat, trans_mat, y, mask, session):
    inputs_obs = []
    inputs_trans = []
    datas = []
    masks = []

    unique_sessions = pd.Series(session).drop_duplicates().to_numpy()

    counter = 0

    for sess in unique_sessions:
        idx = np.where(session == sess)[0]

        inputs_obs.append(obs_mat[idx, :])
        inputs_trans.append(trans_mat[idx, :])
        datas.append(y[idx, :].reshape(-1, 1).astype(int))
        masks.append(mask[idx].reshape(-1, 1).astype(bool))

        counter += len(idx)

    assert counter == obs_mat.shape[0], "Some trials were not assigned to sessions."

    return inputs_obs, inputs_trans, datas, masks


# ------------------------------------------------------------
# 3) Helper: compute held-out log likelihood safely
# ------------------------------------------------------------

def safe_glmhmm_log_probability(model, datas, obs_inputs, trans_inputs, masks):
    try:
        lp = model.log_probability(
            datas,
            transition_input=trans_inputs,
            observation_input=obs_inputs,
            masks=masks
        )
        return float(np.asarray(lp).sum())
    except Exception as e:
        print("List-level log_probability failed.")
        print("Reason:", repr(e))
        print("Trying sequence-by-sequence evaluation.")

    total_lp = 0.0

    for i in range(len(datas)):
        d = datas[i].reshape(-1, 1).astype(int)
        oi = obs_inputs[i]
        ti = trans_inputs[i]
        m = masks[i].reshape(-1, 1).astype(bool)

        try:
            lp_i = model.log_probability(
                d,
                transition_input=ti,
                observation_input=oi,
                masks=m
            )
        except TypeError:
            lp_i = model.log_probability(
                [d],
                transition_input=[ti],
                observation_input=[oi],
                masks=[m]
            )

        total_lp += float(np.asarray(lp_i).sum())

    return total_lp


# ------------------------------------------------------------
# 4) Settings for full K=2 fold 0 run
# ------------------------------------------------------------

K = 2
D = 1
C = 2
fold = 0

M_obs = obs_mat.shape[1]
M_trans = trans_mat.shape[1]

# GitHub global script uses 300 EM iterations.
num_iters_EM_fit = 300

# Start with 3 initializations.
# After this works, we can expand to more initializations and more K values.
iter_ids = [0, 1, 2]

prior_sigma = 100.0
transition_alpha = 1.0
GLM_T_init_0 = True

print("K:", K)
print("fold:", fold)
print("EM iterations:", num_iters_EM_fit)
print("initializations:", iter_ids)
print("M_obs:", M_obs)
print("M_trans:", M_trans)


# ------------------------------------------------------------
# 5) Split training and held-out test sessions
# ------------------------------------------------------------

train_sessions = fold_mapping_session[
    fold_mapping_session[:, 1].astype(int) != fold,
    0
].astype(str)

test_sessions = fold_mapping_session[
    fold_mapping_session[:, 1].astype(int) == fold,
    0
].astype(str)

train_idx = np.isin(session, train_sessions)
test_idx = np.isin(session, test_sessions)

obs_train = obs_mat[train_idx, :]
trans_train = trans_mat[train_idx, :]
y_train = output[train_idx, :]
mask_train = mask[train_idx]
session_train = session[train_idx]

obs_test = obs_mat[test_idx, :]
trans_test = trans_mat[test_idx, :]
y_test = output[test_idx, :]
mask_test = mask[test_idx]
session_test = session[test_idx]

inputs_train, inputs_trans_train, datas_train, masks_train = segment_by_session(
    obs_train,
    trans_train,
    y_train,
    mask_train,
    session_train
)

inputs_test, inputs_trans_test, datas_test, masks_test = segment_by_session(
    obs_test,
    trans_test,
    y_test,
    mask_test,
    session_test
)

print("Training trials:", y_train.shape[0])
print("Held-out test trials:", y_test.shape[0])
print("Training sessions:", len(np.unique(session_train)))
print("Held-out test sessions:", len(np.unique(session_test)))
print("Number of training sequences:", len(datas_train))
print("Number of held-out test sequences:", len(datas_test))


# ------------------------------------------------------------
# 6) Load 1-state GLM initialization for this fold
# ------------------------------------------------------------

glm_init_path = (
    RESULTS_DIR
    / "Model"
    / "glm_#state=1"
    / f"fld_num={fold}"
    / "important_params_iter_0.npz"
)

if not glm_init_path.exists():
    raise FileNotFoundError(f"Could not find GLM initialization file: {glm_init_path}")

glm_container = np.load(glm_init_path, allow_pickle=True)

glm_weights_1state = glm_container["arr_1"]

print("Loaded GLM initialization from:")
print(glm_init_path)
print("Original saved GLM weights shape:", glm_weights_1state.shape)

# Convert 1 x 2 x M_obs into 1 x 1 x M_obs for input_driven_obs_diff_inputs
if glm_weights_1state.shape[1] == 2:
    glm_weights_for_hmm = glm_weights_1state[:, 1:2, :]
elif glm_weights_1state.shape[1] == 1:
    glm_weights_for_hmm = glm_weights_1state
else:
    raise ValueError("Unexpected GLM weight shape.")

print("Corrected GLM weights for HMM shape:", glm_weights_for_hmm.shape)


# ------------------------------------------------------------
# 7) Baseline likelihood for comparison
# ------------------------------------------------------------

p_right_baseline = np.mean(y_train[:, 0])
p_right_baseline = np.clip(p_right_baseline, 1e-12, 1 - 1e-12)

baseline_test_ll = np.sum(
    y_test[:, 0] * np.log(p_right_baseline)
    + (1 - y_test[:, 0]) * np.log(1 - p_right_baseline)
)

print("Baseline p(right):", p_right_baseline)
print("Baseline test LL:", baseline_test_ll)


# ------------------------------------------------------------
# 8) Fit K=2 fold 0 for multiple initializations
# ------------------------------------------------------------

summary_rows = []

for iter_id in iter_ids:

    print("\n================================================")
    print(f"Fitting full GLM-HMM: K={K}, fold={fold}, iter={iter_id}")
    print("================================================")

    # Create model
    this_hmm = ssm.HMM_TO(
        K,
        D,
        M_trans=M_trans,
        M_obs=M_obs,
        observations="input_driven_obs_diff_inputs",
        observation_kwargs=dict(
            C=C,
            prior_sigma=prior_sigma
        ),
        transitions="inputdrivenalt",
        transition_kwargs=dict(
            prior_sigma=prior_sigma,
            alpha=transition_alpha,
            kappa=0
        )
    )

    # Initialize observation weights from 1-state GLM + random noise
    rng = np.random.default_rng(iter_id)

    obs_init = np.tile(glm_weights_for_hmm, (K, 1, 1))
    obs_init = obs_init + rng.normal(0, 0.05, size=obs_init.shape)

    assert obs_init.shape == (K, C - 1, M_obs)

    this_hmm.observations.params = obs_init

    # Initialize transition weights to zero, as in the author's global script setting
    if GLM_T_init_0:
        try:
            this_hmm.transitions.params[1][...] = np.zeros_like(
                this_hmm.transitions.params[1]
            )
            print("Transition GLM weights initialized to zero.")
        except Exception as e:
            print("Could not zero transition weights; continuing.")
            print("Reason:", repr(e))

    # Fit model
    lls = this_hmm.fit(
        datas_train,
        transition_input=inputs_trans_train,
        observation_input=inputs_train,
        train_masks=masks_train,
        method="em",
        num_iters=num_iters_EM_fit,
        initialize=False,
        tolerance=1e-4
    )

    train_initial_ll = float(lls[0])
    train_final_ll = float(lls[-1])
    train_ll_improvement = train_final_ll - train_initial_ll

    # Evaluate held-out sessions
    test_ll = safe_glmhmm_log_probability(
        this_hmm,
        datas_test,
        inputs_test,
        inputs_trans_test,
        masks_test
    )

    test_n_trials = int(sum(d.shape[0] for d in datas_test))
    test_ll_per_trial = test_ll / test_n_trials

    bits_per_trial = (test_ll - baseline_test_ll) / (
        test_n_trials * np.log(2)
    )

    print("Initial train LL:", train_initial_ll)
    print("Final train LL:", train_final_ll)
    print("Train LL improvement:", train_ll_improvement)
    print("Test LL:", test_ll)
    print("Test LL per trial:", test_ll_per_trial)
    print("Bits per trial over baseline:", bits_per_trial)

    # Save model
    model_dir = (
        RESULTS_DIR
        / "Model"
        / f"glmhmm_#state={K}"
        / f"fld_num={fold}"
        / f"iter_{iter_id}"
    )

    model_dir.mkdir(parents=True, exist_ok=True)

    raw_param_path = model_dir / f"glm_hmm_raw_parameters_itr_{iter_id}.npz"

    np.savez(
        raw_param_path,
        params=np.array(this_hmm.params, dtype=object),
        train_lls=np.asarray(lls)
    )

    summary_rows.append({
        "K": K,
        "fold": fold,
        "iter": iter_id,
        "num_iters_EM_fit": num_iters_EM_fit,
        "n_train_trials": y_train.shape[0],
        "n_test_trials": test_n_trials,
        "n_train_sessions": len(np.unique(session_train)),
        "n_test_sessions": len(np.unique(session_test)),
        "initial_train_ll": train_initial_ll,
        "final_train_ll": train_final_ll,
        "ll_improvement": train_ll_improvement,
        "test_ll": test_ll,
        "test_ll_per_trial": test_ll_per_trial,
        "baseline_test_ll": baseline_test_ll,
        "bits_per_trial_over_baseline": bits_per_trial,
        "raw_param_path": str(raw_param_path)
    })


# ------------------------------------------------------------
# 9) Save and display K=2 fold 0 full-run summary
# ------------------------------------------------------------

full_summary = pd.DataFrame(summary_rows)

summary_path = (
    RESULTS_DIR
    / "Model"
    / f"glmhmm_#state={K}"
    / f"fld_num={fold}"
    / f"glmhmm_full_summary_K{K}_fold{fold}.csv"
)

full_summary.to_csv(summary_path, index=False)

print("\nFull K=2 fold 0 summary:")
display(full_summary)

print("\nSaved full-run summary to:")
print(summary_path)

best_row = full_summary.loc[full_summary["test_ll"].idxmax()]

print("\nBest initialization by held-out test LL:")
display(best_row.to_frame().T)

obs_mat: (12150, 5)
trans_mat: (12150, 6)
output: (12150, 1)
session: (12150,)
fold mapping: (405, 2)
K: 2
fold: 0
EM iterations: 300
initializations: [0, 1, 2]
M_obs: 5
M_trans: 6
Training trials: 9450
Held-out test trials: 2700
Training sessions: 315
Held-out test sessions: 90
Number of training sequences: 315
Number of held-out test sequences: 90
Loaded GLM initialization from:
/content/wm_glmhmm_results/Model/glm_#state=1/fld_num=0/important_params_iter_0.npz
Original saved GLM weights shape: (1, 2, 5)
Corrected GLM weights for HMM shape: (1, 1, 5)
Baseline p(right): 0.48074074074074075
Baseline test LL: -1871.1125075402192

Fitting full GLM-HMM: K=2, fold=0, iter=0
Transition GLM weights initialized to zero.


  0%|          | 0/300 [00:00<?, ?it/s]

Initial train LL: -8722.475383310632
Final train LL: -5793.674676478252
Train LL improvement: 2928.8007068323795
Test LL: -1691.6567293768794
Test LL per trial: -0.6265395293988443
Bits per trial over baseline: 0.09588887452411861

Fitting full GLM-HMM: K=2, fold=0, iter=1
Transition GLM weights initialized to zero.


  0%|          | 0/300 [00:00<?, ?it/s]

Initial train LL: -8724.527178554925
Final train LL: -5793.679411121946
Train LL improvement: 2930.847767432979
Test LL: -1691.694229849903
Test LL per trial: -0.626553418462927
Bits per trial over baseline: 0.09586883684024378

Fitting full GLM-HMM: K=2, fold=0, iter=2
Transition GLM weights initialized to zero.


  0%|          | 0/300 [00:00<?, ?it/s]

Initial train LL: -8758.49468003791
Final train LL: -5793.621112000926
Train LL improvement: 2964.8735680369837
Test LL: -1691.6730957172808
Test LL per trial: -0.6265455910064003
Bits per trial over baseline: 0.09588012947295764

Full K=2 fold 0 summary:


,K,fold,iter,num_iters_EM_fit,n_train_trials,n_test_trials,n_train_sessions,n_test_sessions,initial_train_ll,final_train_ll,ll_improvement,test_ll,test_ll_per_trial,baseline_test_ll,bits_per_trial_over_baseline,raw_param_path
0,2,0,0,300,9450,2700,315,90,-8722.475383,-5793.674676,2928.800707,-1691.656729,-0.626540,-1871.112508,0.095889,/content/wm_glmhmm_results/Model/glmhmm_#state...
1,2,0,1,300,9450,2700,315,90,-8724.527179,-5793.679411,2930.847767,-1691.694230,-0.626553,-1871.112508,0.095869,/content/wm_glmhmm_results/Model/glmhmm_#state...
2,2,0,2,300,9450,2700,315,90,-8758.494680,-5793.621112,2964.873568,-1691.673096,-0.626546,-1871.112508,0.095880,/content/wm_glmhmm_results/Model/glmhmm_#state...



Saved full-run summary to:
/content/wm_glmhmm_results/Model/glmhmm_#state=2/fld_num=0/glmhmm_full_summary_K2_fold0.csv

Best initialization by held-out test LL:


,K,fold,iter,num_iters_EM_fit,n_train_trials,n_test_trials,n_train_sessions,n_test_sessions,initial_train_ll,final_train_ll,ll_improvement,test_ll,test_ll_per_trial,baseline_test_ll,bits_per_trial_over_baseline,raw_param_path
0,2,0,0,300,9450,2700,315,90,-8722.475383,-5793.674676,2928.800707,-1691.656729,-0.62654,-1871.112508,0.095889,/content/wm_glmhmm_results/Model/glmhmm_#state...
